In [0]:
storage_account_name = "librarydatastorage2"
storage_account_key = "<YOUR_STORAGE_KEY>"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
train_cleaned_path = "abfss://raw@librarydatastorage2.dfs.core.windows.net/processed/train_cleaned"
weather_path = "abfss://raw@librarydatastorage2.dfs.core.windows.net/weather_train.csv"
building_path = "abfss://raw@librarydatastorage2.dfs.core.windows.net/building_metadata.csv"

train_df = spark.read.parquet(train_cleaned_path)
weather_df = spark.read.csv(weather_path, header=True, inferSchema=True)
building_df = spark.read.csv(building_path, header=True, inferSchema=True)

In [0]:
from pyspark.sql.functions import to_timestamp

weather_df = weather_df.dropna()

weather_df = weather_df.withColumn(
    "timestamp", to_timestamp("timestamp")
)

building_df = building_df.dropna()

In [0]:
df_joined = train_df.join(
    building_df,
    on="building_id",
    how="left"
)

In [0]:
df_joined = df_joined.join(
    weather_df,
    on=["site_id", "timestamp"],
    how="left"
)

In [0]:
df_joined.printSchema()
df_joined.show(5)

root
 |-- site_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- meter: integer (nullable = true)
 |-- meter_reading: double (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: double (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- floor_count: double (nullable = true)
 |-- air_temperature: double (nullable = true)
 |-- cloud_coverage: double (nullable = true)
 |-- dew_temperature: double (nullable = true)
 |-- precip_depth_1_hr: double (nullable = true)
 |-- sea_level_pressure: double (nullable = true)
 |-- wind_direction: double (nullable = true)
 |-- wind_speed: double (nullable = true)

+-------+-------------------+-----------+-----+-------------+-----------+-----------+----------+-----------+---------------+--------------+---------------+-----------------+------------------+--------------+----------+
|site_id|          timestamp|building_id|meter|meter_reading|prim

In [0]:
output_path = "abfss://raw@librarydatastorage2.dfs.core.windows.net/processed/enriched_dataset"

df_joined.write.mode("overwrite").parquet(output_path)